## Упражнение 04. A/B-тестирование

In [1]:
import pandas as pd
import sqlite3


### Соединение с базой данных

In [2]:
connection = sqlite3.connect('data/checking-logs.sqlite')


### Запрос: средняя дельта "до" и "после" первого визита в ленту, для пользователей, у которых есть наблюдения и до, и после (без project1)

In [3]:
def ab_query(table):
    return f"""
    WITH deltas AS (
        SELECT t.uid AS uid,
               CASE 
                   WHEN t.first_commit_ts < t.first_view_ts THEN 'before' 
                   WHEN t.first_commit_ts >= t.first_view_ts THEN 'after'
               END AS time,
               (deadlines.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0 AS diff
        FROM {table} t
        JOIN deadlines ON t.labname = deadlines.labs
        WHERE t.labname != 'project1'
    ),
    valid_uids AS (
        SELECT uid FROM deltas GROUP BY uid HAVING COUNT(DISTINCT time) = 2
    )
    SELECT time, AVG(diff) AS avg_diff
    FROM deltas
    WHERE uid IN (SELECT uid FROM valid_uids)
    GROUP BY time;
    """


### test_results

In [4]:
test_results = pd.io.sql.read_sql(ab_query('test'), connection)
test_results


,time,avg_diff
0,after,105.229241
1,before,61.156632


### control_results

In [5]:
control_results = pd.io.sql.read_sql(ab_query('control'), connection)
control_results


,time,avg_diff
0,after,118.144571
1,before,99.901448


### Закрытие соединения

In [6]:
connection.close()


### Вывод

В группе test средняя дельта между первым коммитом и дедлайном увеличилась с примерно 61 часа до первого визита до примерно 105 часов после первого визита. Изменение составило около 44 часов.

В группе control средняя дельта также увеличилась - примерно с 100 до 118 часов, то есть примерно на 18 часов. При этом для контрольной группы граница before/after определяется относительно средней даты первого визита из группы test, а не относительно реального визита пользователя.

Таким образом, увеличение средней дельты в группе test оказалось больше, чем в группе control (примерно 44 часа против 18 часов). Наблюдаемая разница согласуется с гипотезой, что после появления страницы "Лента" пользователи могли начинать работу над лабораторными раньше относительно дедлайна.